Hacer un codigo que permita crear en el disco duro un archivo con una matriz de 100.000x100.000 con  ceros y unos como entradas de la matriz

In [5]:
#  MATRIZ BINARIA 100,000 x 100,000
import numpy as np
import os

FILAS    = 100000   # filas de la matriz
COLUMNAS = 100000   # columnas de la matriz

# Para saber cuántos bytes necesita UNA fila de 100,000 columnas:
# 100,000 ÷ 8 = 12,500 bytes exactos por fila
BYTES_POR_FILA = COLUMNAS // 8

# Para crear la matriz sin saturar la ram, la vamos creando por partes (chunks)
# Cada chunk procesará 1000 filas a la vez, lo que es manejable para la RAM
CHUNK = 1000

# Nombre del archivo
NOMBRE_ARCHIVO = "mi_matriz.bin"
print("Creando matriz binaria en el disco...")

# Calculamos el tamaño total en gigabytes
total_bytes = FILAS * BYTES_POR_FILA          # total de bytes del archivo
total_gb    = total_bytes / 1_000_000_000     # convertir a GB (÷ 1 billón)

print(f"Tamaño de la matriz : {FILAS:,} x {COLUMNAS:,}")
print(f"Tamaño en disco     : {total_gb:.2f} GB")

# np.memmap es como abrir una "ventana" al disco duro
# En vez de cargar todo en RAM, escribe directamente al disco
# dtype=np.uint8 cada celda guardará un número entre 0 y 255 (1 byte)
# mode='w+' crear el archivo para escritura (si ya existe, lo borra)
# shape=(...) es la forma, FILAS filas, BYTES_POR_FILA columnas de bytes
archivo_disco = np.memmap(
    NOMBRE_ARCHIVO,
    dtype=np.uint8,
    mode='w+',
    shape=(FILAS, BYTES_POR_FILA)
)
# En este momento el archivo ya existe en el disco duro, pero está vacío (lleno de ceros)

# Ahora vamos a llenar el archivo con datos aleatorios, pero lo haremos por partes (chunks)
# "range(0, FILAS, CHUNK)" genera números: 0, 1000, 2000, 3000, ..., 99000
# Es decir, el número de fila donde EMPIEZA cada chunk

np.random.seed(42)  # Usamos una semilla para que los números aleatorios sean siempre los mismos

for fila_inicio in range(0, FILAS, CHUNK):
    # Calcula dónde termina este chunk: fila_inicio + CHUNK
    # min() se asegura de que no pasemos de la última fila (FILAS)
    fila_fin = min(fila_inicio + CHUNK, FILAS)

    # Cuantas filas tiene este chunk (puede ser menos de CHUNK en el último bloque)
    n_filas = fila_fin - fila_inicio

    # Generar un bloque de datos aleatorios: matriz de 0s y 1s de tamaño (n_filas x COLUMNAS)
    # np.random.randint(0, 2) genera números aleatorios entre 0 y 1 (excluyendo el 2)
    bloque = np.random.randint(0, 2, size=(n_filas, COLUMNAS), dtype=np.uint8)

    # Comprimimos el bloque de bits en bytes usando np.packbits
    # Cada grupo de 8 bits (0s y 1s) se convierte en un byte (0-255)
    bloque_comprimido = np.packbits(bloque, axis=1)

    # Escribimos el bloque comprimido al archivo en disco en la posición correcta
    archivo_disco[fila_inicio:fila_fin] = bloque_comprimido

    # Flush asegura que los datos se escriban físicamente al disco duro, no solo en la caché
    archivo_disco.flush()

    # Mostramos el progreso cada 10,000 filas para no saturar la consola
    if fila_inicio % 10_000 == 0:
        porcentaje = fila_fin / FILAS * 100
        print(f"  Progreso: {porcentaje:.0f}% (fila {fila_fin:,} de {FILAS:,})")


# Cuando terminamos de escribir, cerramos el archivo (liberamos la "ventana" al disco)
del archivo_disco

# Creamos un archivo de texto para guardar los metadatos y la descripción de cómo se guardó la matriz
NOMBRE_META = "mi_matriz.meta"

with open(NOMBRE_META, 'w') as meta:

    meta.write("METADATOS DE LA MATRIZ BINARIA\n")

    meta.write("[DIMENSIONES]\n")
    meta.write(f"filas               = {FILAS}\n")
    meta.write(f"columnas            = {COLUMNAS}\n")
    meta.write(f"total de valores    = {FILAS * COLUMNAS:,}\n\n")

    meta.write("[FORMATO DE ALMACENAMIENTO]\n")
    meta.write(f"tipo de valores     = binario (solo 0s y 1s)\n")
    meta.write(f"empaquetado         = 8 bits por byte\n")
    meta.write(f"bytes por fila      = {BYTES_POR_FILA}\n")
    meta.write(f"tipo de dato        = uint8 (entero sin signo de 8 bits)\n\n")

    tamaño_bytes = os.path.getsize(NOMBRE_ARCHIVO)
    meta.write("[TAMAÑO EN DISCO]\n")
    meta.write(f"megabytes           = {tamaño_bytes / 1e6:.2f} MB\n")
    meta.write(f"gigabytes           = {tamaño_bytes / 1e9:.4f} GB\n\n")

    # Instrucciones para leerlo con Python
    meta.write("[CÓMO LEER ESTE ARCHIVO EN PYTHON]\n")
    meta.write("  import numpy as np\n")
    meta.write(f"  datos = np.memmap('mi_matriz.bin', dtype=np.uint8,\n")
    meta.write(f"                     mode='r', shape=({FILAS}, {BYTES_POR_FILA}))\n")
    meta.write(f"  fila_completa = np.unpackbits(datos[0])[:COLUMNAS]\n\n")

print(f"Metadatos guardados en: {NOMBRE_META}")

# Luego podemos verificar el tamaño real del archivo en el disco duro usando os.path.getsize
print("\nArchivo creado: ", NOMBRE_ARCHIVO)
print(f"Tamaño real: {os.path.getsize(NOMBRE_ARCHIVO) / 1e9:.2f} GB")

# Mostramos los 100 primeros bits y los últimos 100 bits para verificar que se guardó correctamente
# Abrimos el archivo completo en modo lectura
datos = np.memmap(NOMBRE_ARCHIVO, dtype=np.uint8, mode='r',
                  shape=(FILAS, BYTES_POR_FILA))

# PRIMEROS 100 NÚMEROS
# La matriz está guardada fila por fila, entonces los primeros
# 100 valores están al inicio de la primera fila (fila 0)
primera_fila = np.unpackbits(datos[0])
primeros_100 = primera_fila[:100]

print("PRIMEROS 100 NÚMEROS DE LA MATRIZ:")
print(primeros_100.tolist())

# ÚLTIMOS 100 NÚMEROS
ultima_fila  = np.unpackbits(datos[FILAS - 1])
ultimos_100  = ultima_fila[COLUMNAS - 100 : COLUMNAS]

print("  ÚLTIMOS 100 NÚMEROS DE LA MATRIZ:")
print(ultimos_100.tolist())
# Cerrar el archivo
del datos

Creando matriz binaria en el disco...
Tamaño de la matriz : 100,000 x 100,000
Tamaño en disco     : 1.25 GB
  Progreso: 1% (fila 1,000 de 100,000)
  Progreso: 11% (fila 11,000 de 100,000)
  Progreso: 21% (fila 21,000 de 100,000)
  Progreso: 31% (fila 31,000 de 100,000)
  Progreso: 41% (fila 41,000 de 100,000)
  Progreso: 51% (fila 51,000 de 100,000)
  Progreso: 61% (fila 61,000 de 100,000)
  Progreso: 71% (fila 71,000 de 100,000)
  Progreso: 81% (fila 81,000 de 100,000)
  Progreso: 91% (fila 91,000 de 100,000)
Metadatos guardados en: mi_matriz.meta

Archivo creado:  mi_matriz.bin
Tamaño real: 1.25 GB
PRIMEROS 100 NÚMEROS DE LA MATRIZ:
[0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1]
  ÚLTIMOS 100 NÚMEROS DE LA MATRIZ:
[0, 0, 0, 0, 1, 1, 0